In [ ]:
import sys
from pathlib import Path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from utils.llm import call_openai, extract_json
from utils.state import PipelineState
import json
from dotenv import load_dotenv
load_dotenv()


In [ ]:
initial: PipelineState = {
        "watchlist":          ['TSLA', 'OKLO'],
        "openai_key": '',
        "query_bundles":      [],
        "market_context":     {},
        "raw_articles":       [],
        "raw_article_count":  0,
        "clean_articles":     [],
        "clean_article_count":0,
        "event_clusters":     [],
        "event_cards":        [],
        "ranked_events":      [],
        "digest":             {},
        "current_step":       0,
        "step_logs":          [],
        "error":              None,
    }


## Agent 1 Watchlist

In [ ]:
"""Agent 1: Watchlist & Context — resolves tickers to query bundles."""
PROMPT = """Given these stock tickers, create a query bundle for each.

Return a JSON array. Each item:
{{
  "ticker": "AAPL",
  "company_name": "Apple Inc.",
  "aliases": ["Apple", "Apple Inc"],
  "industry": "Consumer Electronics",
  "company_queries": ["Apple earnings Q4 2025", "Apple iPhone revenue", "Apple stock news"],
  "industry_queries": ["smartphone market trends 2025", "consumer electronics industry"]
}}

Tickers: {tickers}

Respond with ONLY the JSON array."""


def watchlist_agent(state: PipelineState) -> PipelineState:
    tickers = state["watchlist"]
    state["current_step"] = 1
    state["step_logs"].append(f"[Agent 1] Expanding {len(tickers)} tickers into query bundles...")

    prompt = PROMPT.format(tickers=", ".join(tickers))
    try:
        response = call_openai(prompt, state["openai_key"])
        bundles = extract_json(response)

        state["query_bundles"] = bundles
        state["step_logs"].append(f"[Agent 1] ✓ Generated {len(bundles)} query bundles")
    except Exception as e:
        state["error"] = f"Agent 1 failed: {e}"
        state["query_bundles"] = []

    return state

def test_watchlist_agent(state: PipelineState) -> PipelineState:
    tickers = state["watchlist"]
    state["current_step"] = 1
    state["step_logs"].append(f"[Agent 1] Expanding {len(tickers)} tickers into query bundles...")

    prompt = PROMPT.format(tickers=", ".join(tickers))
    try:
        response = call_openai(prompt, state["openai_key"])
        bundles = extract_json(response)
        state["query_bundles"] = bundles
        state["step_logs"].append(f"[Agent 1] ✓ Generated {len(bundles)} query bundles")
    except Exception as e:
        state["error"] = f"Agent 1 failed: {e}"
        state["query_bundles"] = []

    return state


In [ ]:
watchlist_agent_test = watchlist_agent(initial)
print(watchlist_agent_test)

In [ ]:
output_path = "agent_1_output.json"
with open(output_path, "w") as f:
    json.dump(watchlist_agent_test, f, indent=4)

## Load Agent 1 output from disk (for testing Agent 1b without hitting APIs)

In [ ]:
with open("agent_1_output.json", "r") as f:
    watchlist_agent_test = json.load(f)

## Agent 1b Market Data

In [ ]:
"""
Agent 1b: MarketDataAgent
Fetches supplementary quantitative market context per ticker.
Runs independently of the article flow and populates state["market_context"].

Output schema per ticker:
  last_price          – Latest close price
  currency            – Trading currency (SGD, USD, etc.)
  price_change_1d     – 1-day price change %
  price_change_5d     – 5-day price change %
  volume_ratio        – Today's volume / 30-day avg volume
  analyst_rating      – Consensus rating string (Buy / Hold / Sell)
  target_price        – Consensus 12-month target price
  earnings_date       – Next earnings date (ISO 8601)
  recent_eps_actual   – Most recent reported EPS
  recent_eps_estimate – Consensus EPS estimate for that period
  fetched_at          – ISO 8601 UTC timestamp of fetch

Note: This implementation uses Yahoo Finance RSS as a free data source.
      Replace _fetch_yahoo() with a premium data provider (Alpha Vantage,
      Polygon.io, Refinitiv) for production use with real numerical data.
"""

import json
import urllib.request
import urllib.parse
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed

from utils.state import PipelineState


YF_QUOTE_URL = "https://query1.finance.yahoo.com/v8/finance/chart/{ticker}?interval=1d&range=5d"
YF_SUMMARY_URL = "https://query2.finance.yahoo.com/v10/finance/quoteSummary/{ticker}?modules=financialData,defaultKeyStatistics,calendarEvents,summaryDetail"


def _fetch_yahoo_chart(ticker: str) -> dict:
    """Fetch 5-day OHLCV from Yahoo Finance chart API."""
    url = YF_QUOTE_URL.format(ticker=urllib.parse.quote(ticker))
    req = urllib.request.Request(url, headers={
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept": "application/json",
    })
    with urllib.request.urlopen(req, timeout=10) as r:
        data = json.loads(r.read())
    result = data["chart"]["result"][0]
    meta   = result["meta"]
    closes = result["indicators"]["quote"][0].get("close", [])
    volumes = result["indicators"]["quote"][0].get("volume", [])
    return {
        "last_price":      meta.get("regularMarketPrice"),
        "currency":        meta.get("currency", ""),
        "prev_close":      meta.get("previousClose") or meta.get("chartPreviousClose"),
        "closes":          [c for c in closes if c is not None],
        "volumes":         [v for v in volumes if v is not None],
        "regular_volume":  meta.get("regularMarketVolume"),
    }


def _fetch_yahoo_summary(ticker: str) -> dict:
    """Fetch analyst consensus and earnings data from Yahoo Finance quoteSummary."""
    url = YF_SUMMARY_URL.format(ticker=urllib.parse.quote(ticker))
    req = urllib.request.Request(url, headers={
        "User-Agent": "Mozilla/5.0",
        "Accept": "application/json",
    })
    with urllib.request.urlopen(req, timeout=10) as r:
        data = json.loads(r.read())
    result = data.get("quoteSummary", {}).get("result", [{}])[0]
    fin    = result.get("financialData", {})
    cal    = result.get("calendarEvents", {})
    return {
        "analyst_rating":      fin.get("recommendationKey", "").title() or None,
        "target_price":        fin.get("targetMeanPrice", {}).get("raw"),
        "recent_eps_actual":   fin.get("earningsPerShare", {}).get("raw"),
        "recent_eps_estimate": None,  # available via earningsTrend; omitted for simplicity
        "earnings_date":       (
            cal.get("earnings", {})
               .get("earningsDate", [{}])[0]
               .get("fmt")
        ),
    }


def _build_market_context_for_ticker(ticker: str) -> dict:
    """Build the full market context dict for one ticker. Returns empty dict on failure."""
    now = datetime.now(timezone.utc).isoformat()
    try:
        chart   = _fetch_yahoo_chart(ticker)
        summary = _fetch_yahoo_summary(ticker)

        closes  = chart["closes"]
        last    = chart["last_price"] or (closes[-1] if closes else None)
        prev    = chart["prev_close"] or (closes[-2] if len(closes) >= 2 else None)

        p1d = round((last / prev - 1) * 100, 2) if last and prev and prev != 0 else None
        p5d = round((last / closes[0] - 1) * 100, 2) if last and closes and closes[0] else None

        volumes = chart["volumes"]
        avg30   = chart["regular_volume"]
        vol_ratio = round(avg30 / (sum(volumes) / len(volumes)), 2) if volumes and avg30 else None

        return {
            "last_price":          last,
            "currency":            chart["currency"],
            "price_change_1d":     p1d,
            "price_change_5d":     p5d,
            "volume_ratio":        vol_ratio,
            "analyst_rating":      summary["analyst_rating"],
            "target_price":        summary["target_price"],
            "earnings_date":       summary["earnings_date"],
            "recent_eps_actual":   summary["recent_eps_actual"],
            "recent_eps_estimate": summary["recent_eps_estimate"],
            "fetched_at":          now,
        }
    except Exception as e:
        # Return a minimal stub so downstream agents degrade gracefully
        return {"fetched_at": now, "_error": str(e)}


def market_data_agent(state: PipelineState) -> PipelineState:
    """
    Agent 1b: fetch market context for every ticker in parallel.
    Failures are silently swallowed — market context is supplementary.
    """
    tickers = state.get("watchlist", [])
    state["step_logs"].append(f"[Agent 1b] Fetching market data for {len(tickers)} tickers...")

    market_context: dict = {}

    with ThreadPoolExecutor(max_workers=1) as executor:
        futures = {executor.submit(_build_market_context_for_ticker, t): t for t in tickers}
        for future in as_completed(futures):
            ticker = futures[future]
            market_context[ticker] = future.result()

    state["market_context"] = market_context

    fetched = sum(1 for v in market_context.values() if not v.get("_error"))
    failed  = len(market_context) - fetched
    msg     = f"[Agent 1b] ✓ Market data fetched for {fetched}/{len(tickers)} tickers"
    if failed:
        msg += f" ({failed} failed — will use article-only signals)"
    state["step_logs"].append(msg)
    return state


In [ ]:
market_data_agent_test = market_data_agent(watchlist_agent_test)
market_data_agent_test

In [ ]:
output_path = "agent_1b_output.json"
with open(output_path, "w") as f:
    json.dump(market_data_agent_test, f, indent=4)

## Load Agent 1b output from disk (for testing Agent 2 without hitting APIs)

In [ ]:
with open("agent_1b_output.json", "r") as f:
    market_data_agent_test = json.load(f)

## Agent 2 News Retrieval Agent

In [ ]:
"""
Agent 2: NewsRetrievalAgent  — NO API KEY REQUIRED
Fetches articles from 5 free sources in parallel using only RSS feeds
and public JSON endpoints.

Sources:
  1. Yahoo Finance RSS       – per-ticker headlines       (query_type: company)
  2. Google News RSS         – company + industry search  (query_type: company / industry)
  3. Reuters RSS             – business / markets feed    (query_type: macro)
  4. Finviz                  – per-ticker news scrape      (query_type: company)
  5. Seeking Alpha RSS       – per-ticker news feed        (query_type: company)

All articles are normalised to the canonical schema:
  ticker, company, headline, snippet, url, source,
  published_at, query_type, credibility, raw
"""

import re
import html
import urllib.request
import urllib.parse
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from xml.etree import ElementTree as ET

from utils.state import PipelineState


# ── Credibility tiers ─────────────────────────────────────────────────────────
SOURCE_CREDIBILITY: dict[str, float] = {
    # Tier 1
    "sgx":              1.00,
    "mas":              1.00,
    # Tier 2
    "reuters":          0.85,
    "bloomberg":        0.85,
    "business times":   0.85,
    "straits times":    0.85,
    "ft.com":           0.85,
    "financial times":  0.85,
    "wsj":              0.85,
    "wall street":      0.85,
    # Tier 3
    "cnbc":             0.70,
    "cna":              0.70,
    "nikkei":           0.70,
    "barron":           0.70,
    "marketwatch":      0.70,
    "seeking alpha":    0.70,
    # Tier 4
    "yahoo":            0.55,
    "finviz":           0.55,
    "benzinga":         0.55,
    "motley fool":      0.55,
    "investing.com":    0.55,
}
DEFAULT_CRED = 0.55


def _credibility(source_name: str) -> float:
    sl = source_name.lower()
    for key, score in SOURCE_CREDIBILITY.items():
        if key in sl:
            return score
    return DEFAULT_CRED


# ── Shared HTTP helper ─────────────────────────────────────────────────────────
_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept": "application/rss+xml, application/xml, text/xml, */*",
    "Accept-Language": "en-US,en;q=0.9",
}


def _get(url: str, timeout: int = 12) -> bytes:
    req = urllib.request.Request(url, headers=_HEADERS)
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read()


def _clean(text: str) -> str:
    """Strip HTML tags and decode entities."""
    text = html.unescape(text or "")
    text = re.sub(r"<[^>]+>", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def _parse_rss_date(date_str: str) -> str:
    """Parse RSS pubDate → ISO 8601 UTC. Returns empty string on failure."""
    if not date_str:
        return ""
    for fmt in (
        "%a, %d %b %Y %H:%M:%S %z",
        "%a, %d %b %Y %H:%M:%S GMT",
        "%Y-%m-%dT%H:%M:%S%z",
        "%Y-%m-%dT%H:%M:%SZ",
    ):
        try:
            dt = datetime.strptime(date_str.strip(), fmt)
            if dt.tzinfo is None:
                dt = dt.replace(tzinfo=timezone.utc)
            return dt.astimezone(timezone.utc).isoformat()
        except ValueError:
            continue
    return date_str


def _normalise(
    headline: str,
    snippet: str,
    url: str,
    source: str,
    published_at: str,
    query_type: str,
    ticker: str,
    company: str,
    raw: dict,
) -> dict | None:
    url = (url or "").strip()
    headline = _clean(headline)
    if not url or not headline:
        return None
    return {
        "ticker":       ticker,
        "company":      company,
        "headline":     headline,
        "snippet":      _clean(snippet),
        "url":          url,
        "source":       source,
        "published_at": _parse_rss_date(published_at),
        "query_type":   query_type,
        "credibility":  _credibility(source),
        "raw":          raw,
    }


# ── Source 1: Yahoo Finance RSS ───────────────────────────────────────────────
def _fetch_yahoo_finance_rss(ticker: str, company: str) -> list[dict]:
    """
    Yahoo Finance news RSS feed for a specific ticker.
    URL: https://feeds.finance.yahoo.com/rss/2.0/headline?s=TICKER&region=US&lang=en-US
    """
    url = (
        "https://feeds.finance.yahoo.com/rss/2.0/headline?"
        + urllib.parse.urlencode({"s": ticker, "region": "US", "lang": "en-US"})
    )
    try:
        root = ET.fromstring(_get(url))
        items = root.findall(".//item")
        results = []
        for item in items[:15]:
            art = _normalise(
                headline    = item.findtext("title", ""),
                snippet     = item.findtext("description", ""),
                url         = item.findtext("link", ""),
                source      = "Yahoo Finance",
                published_at= item.findtext("pubDate", ""),
                query_type  = "company",
                ticker      = ticker,
                company     = company,
                raw         = {"feed": "yahoo_finance_rss"},
            )
            if art:
                results.append(art)
        return results
    except Exception as e:
        print(f"    [Agent 2] Yahoo Finance RSS failed ({ticker}): {e}")
        return []


# ── Source 2: Google News RSS ─────────────────────────────────────────────────
def _fetch_google_news_rss(query: str, query_type: str, ticker: str, company: str) -> list[dict]:
    """
    Google News RSS search — no API key needed.
    URL: https://news.google.com/rss/search?q=QUERY&hl=en-US&gl=US&ceid=US:en
    """
    url = (
        "https://news.google.com/rss/search?"
        + urllib.parse.urlencode({
            "q":    query,
            "hl":   "en-US",
            "gl":   "US",
            "ceid": "US:en",
        })
    )
    try:
        root  = ET.fromstring(_get(url))
        ns    = {"media": "http://search.yahoo.com/mrss/"}
        items = root.findall(".//item")
        results = []
        for item in items[:12]:
            # Google News URLs are redirect wrappers — extract the source name
            # from the <source> tag when available
            source_el   = item.find("source")
            source_name = source_el.text if source_el is not None else "Google News"

            art = _normalise(
                headline    = item.findtext("title", ""),
                snippet     = item.findtext("description", ""),
                url         = item.findtext("link", ""),
                source      = source_name,
                published_at= item.findtext("pubDate", ""),
                query_type  = query_type,
                ticker      = ticker,
                company     = company,
                raw         = {"feed": "google_news_rss", "query": query},
            )
            if art:
                results.append(art)
        return results
    except Exception as e:
        print(f"    [Agent 2] Google News RSS failed ({query!r}): {e}")
        return []


# ── Source 3: Reuters RSS ─────────────────────────────────────────────────────
# Reuters provides public RSS feeds for their top sections.
REUTERS_FEEDS = {
    "business":  "https://feeds.reuters.com/reuters/businessNews",
    "markets":   "https://feeds.reuters.com/reuters/companyNews",
    "technology":"https://feeds.reuters.com/reuters/technologyNews",
}


def _fetch_reuters_rss(ticker: str, company: str) -> list[dict]:
    """Fetch Reuters business + markets RSS and keep articles mentioning company/ticker."""
    results = []
    keywords = {ticker.lower(), company.lower().split()[0]}  # e.g. {"aapl", "apple"}

    for feed_name, feed_url in REUTERS_FEEDS.items():
        try:
            root  = ET.fromstring(_get(feed_url))
            items = root.findall(".//item")
            for item in items[:30]:
                title   = item.findtext("title", "")
                desc    = item.findtext("description", "")
                text    = (title + " " + desc).lower()
                if not any(kw in text for kw in keywords):
                    continue
                art = _normalise(
                    headline    = title,
                    snippet     = desc,
                    url         = item.findtext("link", ""),
                    source      = "Reuters",
                    published_at= item.findtext("pubDate", ""),
                    query_type  = "company",
                    ticker      = ticker,
                    company     = company,
                    raw         = {"feed": f"reuters_{feed_name}"},
                )
                if art:
                    results.append(art)
        except Exception as e:
            print(f"    [Agent 2] Reuters RSS ({feed_name}) failed: {e}")

    return results


# ── Source 4: Finviz news ─────────────────────────────────────────────────────
def _fetch_finviz(ticker: str, company: str) -> list[dict]:
    """
    Scrape Finviz ticker news table (plain HTML, no JS).
    URL: https://finviz.com/quote.ashx?t=TICKER
    """
    url = f"https://finviz.com/quote.ashx?t={urllib.parse.quote(ticker)}"
    try:
        raw_html = _get(url).decode("utf-8", errors="ignore")
        # Finviz news rows: <a ...>headline</a> in a table with class "news-link"
        pattern = re.compile(
            r'<a[^>]+class="news-link"[^>]+href="([^"]+)"[^>]*>([^<]+)</a>'
            r'.*?<td[^>]*>(\w+\s+\d+[^<]*)</td>',
            re.DOTALL,
        )
        results = []
        for m in pattern.finditer(raw_html):
            article_url, headline, date_str = m.group(1), m.group(2), m.group(3)
            art = _normalise(
                headline    = headline,
                snippet     = "",
                url         = article_url,
                source      = "Finviz",
                published_at= date_str,
                query_type  = "company",
                ticker      = ticker,
                company     = company,
                raw         = {"feed": "finviz"},
            )
            if art:
                results.append(art)
            if len(results) >= 10:
                break
        return results
    except Exception as e:
        print(f"    [Agent 2] Finviz failed ({ticker}): {e}")
        return []


# ── Source 5: Seeking Alpha RSS ───────────────────────────────────────────────
def _fetch_seeking_alpha_rss(ticker: str, company: str) -> list[dict]:
    """
    Seeking Alpha provides public RSS per ticker.
    URL: https://seekingalpha.com/api/sa/combined/{TICKER}.xml
    """
    url = f"https://seekingalpha.com/api/sa/combined/{urllib.parse.quote(ticker)}.xml"
    try:
        root  = ET.fromstring(_get(url))
        items = root.findall(".//item")
        results = []
        for item in items[:10]:
            art = _normalise(
                headline    = item.findtext("title", ""),
                snippet     = item.findtext("description", ""),
                url         = item.findtext("link", ""),
                source      = "Seeking Alpha",
                published_at= item.findtext("pubDate", ""),
                query_type  = "company",
                ticker      = ticker,
                company     = company,
                raw         = {"feed": "seeking_alpha_rss"},
            )
            if art:
                results.append(art)
        return results
    except Exception as e:
        print(f"    [Agent 2] Seeking Alpha RSS failed ({ticker}): {e}")
        return []


# ── Orchestrator ──────────────────────────────────────────────────────────────
def _fetch_all_for_bundle(bundle: dict) -> list[dict]:
    """Fetch from all 5 sources for one query bundle. Returns list of articles."""
    ticker  = bundle.get("ticker", "UNKNOWN")
    company = bundle.get("company_name", ticker)
    results: list[dict] = []

    # Per-ticker sources (always run)
    results += _fetch_yahoo_finance_rss(ticker, company)
    results += _fetch_seeking_alpha_rss(ticker, company)
    results += _fetch_finviz(ticker, company)
    results += _fetch_reuters_rss(ticker, company)

    # Google News — company queries + industry queries
    for q in bundle.get("company_queries", [])[:3]:    # cap at 3 to avoid rate-limiting
        results += _fetch_google_news_rss(q, "company", ticker, company)
    for q in bundle.get("industry_queries", [])[:2]:
        results += _fetch_google_news_rss(q, "industry", ticker, company)

    return results


def retrieval_agent(state: PipelineState) -> PipelineState:
    state["current_step"] = 2
    bundles = state.get("query_bundles", [])
    state["step_logs"].append(
        f"[Agent 2] Fetching from Yahoo Finance RSS, Google News RSS, Reuters RSS, "
        f"Finviz, Seeking Alpha for {len(bundles)} tickers..."
    )

    seen_urls:    set[str]  = set()
    all_articles: list[dict] = []

    # Fetch all bundles in parallel
    with ThreadPoolExecutor(max_workers=min(len(bundles) * 6, 12)) as executor:
        futures = {executor.submit(_fetch_all_for_bundle, b): b for b in bundles}
        for future in as_completed(futures):
            try:
                for art in future.result():
                    if art["url"] not in seen_urls:
                        seen_urls.add(art["url"])
                        all_articles.append(art)
            except Exception as e:
                ticker = futures[future].get("ticker", "?")
                print(f"    [Agent 2] Bundle fetch error ({ticker}): {e}")

    # Sort newest first
    all_articles.sort(key=lambda a: a.get("published_at", ""), reverse=True)

    # Count by source for logging
    source_counts: dict[str, int] = {}
    for art in all_articles:
        src = art["source"]
        source_counts[src] = source_counts.get(src, 0) + 1

    source_summary = ", ".join(
        f"{src}: {n}" for src, n in sorted(source_counts.items(), key=lambda x: -x[1])
    )

    state["raw_articles"]      = all_articles
    state["raw_article_count"] = len(all_articles)
    state["step_logs"].append(
        f"[Agent 2] ✓ Retrieved {len(all_articles)} raw articles "
        f"({source_summary})"
    )
    return state


In [ ]:
retrieval_agent_2 = retrieval_agent(market_data_agent_test)
retrieval_agent_2

In [ ]:
output_path = "agent_2_output.json"
with open(output_path, "w") as f:
    json.dump(retrieval_agent_2, f, indent=4)

## Load Agent 2 from file


In [ ]:
with open("agent_2_output.json", "r") as f:
    retrieval_agent_2 = json.load(f)

## Agent 3 Filter Agent


In [ ]:
"""
Agent 3: NoiseFilterAgent
Two-pass deduplication and quality filtering as per the data flow spec.

Pass 1 — Hard filters (heuristic, no LLM):
  - Exact URL dedup
  - Snippet length < 20 chars → drop
  - Age > lookback_hours → drop
  - Exact headline hash dedup

Pass 2 — Semantic deduplication (GPT-4o):
  - Near-duplicate headlines (same event, different phrasing) → keep highest-tier source
  - Low-signal content (roundups, clickbait, irrelevant) → drop

Output: same canonical article schema, fewer articles.
"""

import hashlib
from datetime import datetime, timedelta, timezone

from utils.llm import call_openai, extract_json
from utils.state import PipelineState


LOOKBACK_HOURS = 7 * 24   # 7 days default

DEDUP_PROMPT = """You are a financial news editor reviewing articles for a Singapore investor intelligence system.

Return the indices (0-based) of articles to KEEP.

DROP if:
- Near-duplicate of another article (same event reported by multiple sources — keep the one with the most informative snippet, or the highest-credibility source)
- Generic market roundup with no company-specific signal
- Clickbait or irrelevant to financial investing
- Snippet is vague with no factual content

KEEP if:
- Contains specific financial data (earnings, guidance, M&A, regulatory action, analyst rating, leadership change, product launch)
- Source is authoritative (SGX announcements, major business press)
- Unique angle not covered by other articles in the list

Return ONLY a JSON array of integer indices. Example: [0, 2, 5, 8]

Articles:
{articles_text}"""


def _pass1_hard_filter(articles: list[dict], lookback_hours: int = LOOKBACK_HOURS) -> list[dict]:
    """Hard filters: URL dedup, snippet length, recency, headline hash."""
    seen_urls:      set[str] = set()
    seen_headlines: set[str] = set()
    cutoff = datetime.now(timezone.utc) - timedelta(hours=lookback_hours)
    out: list[dict] = []

    for art in articles:
        url      = (art.get("url") or "").strip()
        headline = (art.get("headline") or "").strip()
        snippet  = (art.get("snippet") or "").strip()

        # 1. URL dedup
        if not url or url in seen_urls:
            continue
        seen_urls.add(url)

        # 2. Snippet length
        if len(snippet) < 20:
            continue

        # 3. Recency — parse published_at
        pub_str = art.get("published_at", "")
        if pub_str:
            try:
                pub_dt = datetime.fromisoformat(pub_str.replace("Z", "+00:00"))
                if pub_dt < cutoff:
                    continue
            except ValueError:
                pass   # unparseable timestamp — keep article

        # 4. Exact headline dedup (case-insensitive MD5)
        h_key = hashlib.md5(headline.lower().encode()).hexdigest()
        if h_key in seen_headlines:
            continue
        seen_headlines.add(h_key)

        out.append(art)

    return out


def _pass2_semantic_dedup(articles: list[dict], openai_key: str) -> list[dict]:
    """GPT-4o semantic deduplication in batches of 25."""
    if not articles:
        return []

    kept: list[dict] = []
    batch_size = 25

    for i in range(0, len(articles), batch_size):
        batch = articles[i : i + batch_size]
        articles_text = "\n".join(
            f"[{j}] [cred:{a.get('credibility', 0.55):.2f}] {a['headline']} | {a['snippet'][:120]}"
            for j, a in enumerate(batch)
        )
        try:
            resp    = call_openai(DEDUP_PROMPT.format(articles_text=articles_text), openai_key)
            indices = extract_json(resp)
            kept.extend(batch[idx] for idx in indices if 0 <= idx < len(batch))
        except Exception as e:
            print(f"    [Agent 3] Semantic dedup failed for batch {i//batch_size}: {e} — keeping all")
            kept.extend(batch)

    return kept


def filter_agent(state: PipelineState) -> PipelineState:
    state["current_step"] = 3
    state["step_logs"].append("[Agent 3] Running Pass 1 (hard filters)...")

    after_pass1 = _pass1_hard_filter(state.get("raw_articles", []))
    state["step_logs"].append(
        f"[Agent 3] Pass 1 complete: {state['raw_article_count']} → {len(after_pass1)} articles"
    )

    state["step_logs"].append("[Agent 3] Running Pass 2 (semantic dedup via GPT-4o)...")
    after_pass2 = _pass2_semantic_dedup(after_pass1, state["openai_key"])
    state["clean_articles"]      = after_pass2
    state["clean_article_count"] = len(after_pass2)
    state["step_logs"].append(
        f"[Agent 3] ✓ Pass 2 complete: {len(after_pass1)} → {len(after_pass2)} articles retained"
    )
    return state


In [ ]:
filter_agent_3 = filter_agent(retrieval_agent_2)
filter_agent_3

In [ ]:
output_path = "agent_3_output.json"
with open(output_path, "w") as f:
    json.dump(filter_agent_3, f, indent=4)

In [ ]:
import hashlib
import time
from datetime import datetime, timedelta, timezone
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from utils.llm import call_openai, extract_json
from utils.state import PipelineState

# Constants
LOOKBACK_HOURS = 7 * 24
SIMILARITY_THRESHOLD = 0.85
LLM_BATCH_SIZE = 25

DEDUP_PROMPT = """You are a financial news editor reviewing articles for a Singapore investor intelligence system.
Return the indices (0-based) of articles to KEEP.

DROP if:
- Near-duplicate of another article (keep the most informative or highest-credibility source)
- Generic market roundup with no company-specific signal
- Clickbait or irrelevant to financial investing
- Snippet is vague with no factual content

KEEP if:
- Contains specific financial data (earnings, guidance, M&A, regulatory action, analyst rating, etc.)
- Source is authoritative
- Unique angle not covered by other articles in the list

Return ONLY a JSON array of integer indices. Example: [0, 2, 5, 8]

Articles:
{articles_text}"""

def get_source_tier(source: str) -> float:
    """Combines credibility logic from both snippets."""
    source_lower = (source or "").lower()
    tiers = {
        1.0: ["sgx", "mas", "official", "government"],
        0.85: ["reuters", "bloomberg", "business times", "straits times", "wsj"],
        0.70: ["cna", "nikkei", "cnbc", "financial times"],
        0.55: ["yahoo finance", "google news", "reddit"]
    }
    for score, keywords in tiers.items():
        if any(k in source_lower for k in keywords):
            return score
    return 0.50

def filter_agent2(state: PipelineState) -> PipelineState:
    start_time = time.time()
    state["current_step"] = 3
    state["current_agent"] = "Agent 3: Noise Filtering"
    
    raw_articles = state.get("raw_articles", [])
    initial_count = len(raw_articles)
    
    # --- PASS 1: Hard Filters (Heuristics) ---
    seen_urls = set()
    seen_headlines = set()
    cutoff = datetime.now(timezone.utc) - timedelta(hours=LOOKBACK_HOURS)
    
    pass1_output = []
    for art in raw_articles:
        # Extract fields safely (handles both dict and object notation if needed)
        url = (art.get("url") if isinstance(art, dict) else getattr(art, 'url', "")).strip()
        headline = (art.get("headline") if isinstance(art, dict) else getattr(art, 'headline', "")).strip()
        snippet = (art.get("snippet") if isinstance(art, dict) else getattr(art, 'snippet', "")).strip()
        pub_at = art.get("published_at") if isinstance(art, dict) else getattr(art, 'published_at', "")

        # Dedupe/Length checks
        if not url or url in seen_urls or len(snippet) < 20:
            continue
        
        # Headline hash check
        h_key = hashlib.md5(headline.lower().encode()).hexdigest()
        if h_key in seen_headlines:
            continue
            
        # Recency check
        try:
            pub_dt = datetime.fromisoformat(pub_at.replace("Z", "+00:00"))
            if pub_dt < cutoff:
                continue
        except:
            pass # Keep if date is unparseable

        seen_urls.add(url)
        seen_headlines.add(h_key)
        pass1_output.append(art)

    # --- PASS 2: Algorithmic Deduplication (TF-IDF) ---
    # Removes high-confidence duplicates before hitting the expensive LLM
    if len(pass1_output) > 1:
        headlines = [a.get("headline", "") if isinstance(a, dict) else a.headline for a in pass1_output]
        try:
            vectorizer = TfidfVectorizer(analyzer="char", ngram_range=(2, 3))
            tfidf_matrix = vectorizer.fit_transform(headlines)
            sim_matrix = cosine_similarity(tfidf_matrix)
            
            to_remove = set()
            for i in range(len(pass1_output)):
                for j in range(i + 1, len(pass1_output)):
                    if sim_matrix[i][j] >= SIMILARITY_THRESHOLD:
                        # Logic: Keep the one with higher source credibility
                        source_i = pass1_output[i].get("source", "") if isinstance(pass1_output[i], dict) else pass1_output[i].source
                        source_j = pass1_output[j].get("source", "") if isinstance(pass1_output[j], dict) else pass1_output[j].source
                        if get_source_tier(source_i) >= get_source_tier(source_j):
                            to_remove.add(j)
                        else:
                            to_remove.add(i)
            
            pass2_output = [a for i, a in enumerate(pass1_output) if i not in to_remove]
        except Exception as e:
            print(f"Pass 2 Vectorization failed: {e}")
            pass2_output = pass1_output
    else:
        pass2_output = pass1_output

    # --- PASS 3: Semantic Filtering (LLM) ---
    # Final check for relevance and complex near-duplicates
    final_output = []
    for i in range(0, len(pass2_output), LLM_BATCH_SIZE):
        batch = pass2_output[i : i + LLM_BATCH_SIZE]
        formatted_text = ""
        for j, a in enumerate(batch):
            h = a.get("headline") if isinstance(a, dict) else a.headline
            s = a.get("snippet") if isinstance(a, dict) else a.snippet
            src = a.get("source") if isinstance(a, dict) else a.source
            formatted_text += f"[{j}] [Source: {src}] {h} | {s[:120]}\n"

        try:
            resp = call_openai(DEDUP_PROMPT.format(articles_text=formatted_text), state["openai_key"])
            indices = extract_json(resp)
            final_output.extend(batch[idx] for idx in indices if 0 <= idx < len(batch))
        except Exception as e:
            print(f"Pass 3 LLM failed for batch {i}: {e}")
            final_output.extend(batch)

    # Update State
    state["clean_articles"] = final_output
    state["clean_article_count"] = len(final_output)
    state["step_logs"].append(
        f"[Agent 3] Filtered {initial_count} -> {len(pass1_output)} (Hard) -> "
        f"{len(pass2_output)} (Vector) -> {len(final_output)} (LLM)"
    )
    #state["agent_timings"]["agent_3"] = time.time() - start_time
    return state

In [ ]:
filter_agent_3_test2 = filter_agent2(retrieval_agent_2)
filter_agent_3_test2    

In [ ]:
output_path = "agent_3a_output.json"
with open(output_path, "w") as f:
    json.dump(filter_agent_3_test2, f, indent=4)

In [ ]:
"""
base_agent.py
-------------
Abstract base class for all agents in the pipeline.
Each agent must implement the `run()` method.
"""

from abc import ABC, abstractmethod
from utils.logger import get_logger


class BaseAgent(ABC):
    """
    All pipeline agents inherit from this class.

    Subclasses must implement:
        run(self, input_data) -> output_data
    """

    def __init__(self, config: dict):
        self.config = config
        self.logger = get_logger(self.__class__.__name__)

    @abstractmethod
    def run(self, input_data):
        """Execute the agent's core logic."""
        raise NotImplementedError

    def log_start(self, msg: str = ""):
        self.logger.info(f"[{self.__class__.__name__}] Starting. {msg}")

    def log_done(self, msg: str = ""):
        self.logger.info(f"[{self.__class__.__name__}] Done. {msg}")


"""
noise_filter_agent.py
---------------------
Agent 3 — Noise Filtering & Deduplication

Two-pass filtering:
  Pass 1 — Hard filters (URL dedup, snippet length, lookback window)
  Pass 2 — Semantic deduplication via sentence-transformers embeddings

sentence-transformers is imported lazily inside the function so that
NumPy/TensorFlow version conflicts in the host environment do not crash
the process at startup — it degrades gracefully to hard-filter-only mode.
"""

from __future__ import annotations
from datetime import datetime, timedelta, timezone
from core.base_agent import BaseAgent


class NoiseFilterAgent(BaseAgent):

    def __init__(self, config: dict):
        super().__init__(config)
        self.min_snippet_len = config.get("filtering", {}).get("min_snippet_length", 50)
        self.sim_threshold   = config.get("filtering", {}).get("similarity_threshold", 0.85)
        self.lookback_hours  = config.get("retrieval",  {}).get("lookback_hours", 24)
        self.embedding_model = config.get("clustering", {}).get(
            "embedding_model", "sentence-transformers/all-MiniLM-L6-v2"
        )
        self._model           = None   # lazy-loaded
        self._embeddings_ok   = None   # None = not yet tested

    def run(self, raw_articles: list[dict]) -> list[dict]:
        self.log_start(f"Filtering {len(raw_articles)} raw articles.")
        articles = self._hard_filter(raw_articles)
        self.logger.info(f"After hard filters: {len(articles)} articles")
        articles = self._semantic_deduplicate(articles)
        self.logger.info(f"After semantic dedup: {len(articles)} articles")
        self.log_done()
        return articles

    # ── Pass 1: Hard filters ──────────────────────────────────────────────────

    def _hard_filter(self, articles: list[dict]) -> list[dict]:
        cutoff    = datetime.now(timezone.utc) - timedelta(hours=self.lookback_hours)
        seen_urls = set()
        kept      = []

        for a in articles:
            url = a.get("url", "")
            if url and url in seen_urls:
                continue
            if url:
                seen_urls.add(url)

            if len((a.get("snippet") or "").strip()) < self.min_snippet_len:
                continue

            pub = a.get("published_at", "")
            if pub:
                try:
                    dt = datetime.fromisoformat(pub)
                    if dt.tzinfo is None:
                        dt = dt.replace(tzinfo=timezone.utc)
                    if dt < cutoff:
                        continue
                except ValueError:
                    pass

            kept.append(a)
        return kept

    # ── Pass 2: Semantic deduplication ───────────────────────────────────────

    def _semantic_deduplicate(self, articles: list[dict]) -> list[dict]:
        if not self._check_embeddings_available():
            return articles

        by_ticker: dict[str, list[dict]] = {}
        for a in articles:
            by_ticker.setdefault(a.get("ticker", "UNKNOWN"), []).append(a)

        result = []
        for group in by_ticker.values():
            result.extend(self._dedup_group(group))
        return result

    def _dedup_group(self, articles: list[dict]) -> list[dict]:
        if len(articles) <= 1:
            return articles

        try:
            import numpy as np
            model      = self._get_model()
            texts      = [a.get("headline") or (a.get("snippet") or "")[:120] for a in articles]
            embeddings = model.encode(texts, normalize_embeddings=True)

            kept_indices = []
            dropped      = set()

            for i in range(len(articles)):
                if i in dropped:
                    continue
                kept_indices.append(i)
                for j in range(i + 1, len(articles)):
                    if j in dropped:
                        continue
                    sim = float(np.dot(embeddings[i], embeddings[j]))
                    if sim >= self.sim_threshold:
                        if self._source_tier(articles[j]) > self._source_tier(articles[i]):
                            dropped.add(i)
                            kept_indices[-1] = j
                        else:
                            dropped.add(j)

            return [articles[i] for i in dict.fromkeys(kept_indices)]

        except Exception as e:
            self.logger.warning(f"Semantic dedup failed, skipping: {e}")
            return articles

    def _check_embeddings_available(self) -> bool:
        """Test-import sentence_transformers once and cache the result."""
        if self._embeddings_ok is not None:
            return self._embeddings_ok
        try:
            import sentence_transformers  # noqa: F401
            import numpy                  # noqa: F401
            self._embeddings_ok = True
        except Exception as e:
            self.logger.warning(
                f"sentence-transformers unavailable (skipping semantic dedup): {e}\n"
                "To fix: pip install 'numpy<2' then pip install sentence-transformers"
            )
            self._embeddings_ok = False
        return self._embeddings_ok

    def _get_model(self):
        if self._model is None:
            from sentence_transformers import SentenceTransformer
            self.logger.info(f"Loading embedding model: {self.embedding_model}")
            self._model = SentenceTransformer(self.embedding_model)
        return self._model

    @staticmethod
    def _source_tier(article: dict) -> int:
        tiers = {
            "SGX Announcements": 4, "MAS": 4,
            "Business Times": 3, "Straits Times": 3, "Reuters": 3,
            "CNA": 2, "Nikkei Asia": 2, "CNBC Asia": 2, "NewsAPI": 2,
            "Yahoo Finance": 1, "Singapore Business Review": 1,
        }
        return tiers.get(article.get("source", ""), 1)


## Agent 4 Clustering

In [ ]:
"""
Agent 4: EventClusteringAgent
Groups clean articles into event clusters per the data flow spec.

Steps:
  1. Group articles by ticker
  2. Keyword-rule event type classification on headline + snippet
  3. GPT-4o clustering within each ticker group
  4. Select representative article (highest-credibility source per cluster)

Output cluster schema:
  cluster_id              – "{ticker}_c{n:03d}"
  ticker                  – Stock ticker (or "MACRO")
  event_type              – One of the 11 typed event categories
  articles                – List of canonical article dicts
  representative_headline – Headline from the highest-credibility source
  representative_source   – Source name of the representative article
  article_count           – Number of articles in cluster
  sources                 – Deduplicated list of all source names
"""

import re
from utils.llm import call_openai, extract_json
from utils.state import PipelineState


# ── Event type keyword rules (spec Table) ────────────────────────────────────
EVENT_TYPE_RULES: list[tuple[str, list[str]]] = [
    ("earnings_release",  ["earnings", "profit", "revenue", "net income", "eps", "quarterly results", "full-year"]),
    ("dividend",          ["dividend", "distribution", "payout", "dpu"]),
    ("guidance_update",   ["guidance", "outlook", "forecast", "profit warning", "upgrade", "downgrade"]),
    ("ma_announcement",   ["acqui", "merger", "takeover", "buyout", " deal", "bid", "offer", "divest"]),
    ("regulatory_action", ["mas ", "regulation", "compliance", "enforcement", "licence", "fine", "penalty"]),
    ("capital_action",    ["rights issue", "placement", "buyback", "bond issue", "capital raise"]),
    ("leadership_change", ["ceo", "chairman", "board", "appoint", "resign", "retire", "chief executive"]),
    ("litigation",        ["lawsuit", "court", "arbitration", "investigation", "probe", "charges"]),
    ("product_launch",    ["launch", "new product", "partnership", "contract win", "awarded", "signed"]),
    ("analyst_rating",    ["analyst", "target price", "overweight", "underweight", "buy rating", "sell rating", "hold rating", "initiates"]),
    ("general_news",      []),   # catch-all
]


def _classify_event_type(headline: str, snippet: str) -> str:
    text = (headline + " " + snippet).lower()
    for event_type, keywords in EVENT_TYPE_RULES:
        if any(kw in text for kw in keywords):
            return event_type
    return "general_news"


CLUSTER_PROMPT = """You are a financial analyst. The articles below all relate to ticker {ticker}.
Group them into distinct event clusters — each cluster represents exactly ONE real-world event.

Return ONLY a JSON array. Each item:
{{
  "cluster_indices": [0, 2, 5],
  "event_type": "earnings_release"
}}

Valid event_type values:
earnings_release, dividend, guidance_update, ma_announcement, regulatory_action,
capital_action, leadership_change, litigation, product_launch, analyst_rating, general_news

Articles:
{articles_text}"""


def _select_representative(articles: list[dict]) -> tuple[str, str]:
    """Return (headline, source) for the highest-credibility article in a cluster."""
    best = max(articles, key=lambda a: a.get("credibility", 0.0))
    return best.get("headline", ""), best.get("source", "Unknown")


def _cluster_ticker_articles(
    ticker: str,
    articles: list[dict],
    openai_key: str,
    ticker_cluster_offset: int,
) -> list[dict]:
    """Cluster articles for one ticker and return cluster dicts."""
    if not articles:
        return []

    articles_text = "\n".join(
        f"[{i}] {a['headline']} | {a['snippet'][:120]}"
        for i, a in enumerate(articles)
    )

    try:
        resp = call_openai(CLUSTER_PROMPT.format(ticker=ticker, articles_text=articles_text), openai_key)
        raw_clusters = extract_json(resp)
    except Exception as e:
        print(f"    [Agent 4] GPT-4o clustering failed for {ticker}: {e} — one cluster per article")
        raw_clusters = [
            {"cluster_indices": [i], "event_type": _classify_event_type(a["headline"], a["snippet"])}
            for i, a in enumerate(articles)
        ]

    clusters: list[dict] = []
    for n, rc in enumerate(raw_clusters):
        indices = [idx for idx in rc.get("cluster_indices", []) if 0 <= idx < len(articles)]
        if not indices:
            continue

        cluster_articles = [articles[idx] for idx in indices]
        rep_headline, rep_source = _select_representative(cluster_articles)
        all_sources = list({a["source"] for a in cluster_articles})

        # GPT-4o may return an event_type; cross-check with keyword rules
        gpt_event_type = rc.get("event_type", "general_news")
        kw_event_type  = _classify_event_type(rep_headline, cluster_articles[0].get("snippet", ""))
        # Prefer keyword match (deterministic) unless GPT returned a more specific type
        event_type = gpt_event_type if gpt_event_type != "general_news" else kw_event_type

        clusters.append({
            "cluster_id":             f"{ticker}_c{ticker_cluster_offset + n:03d}",
            "ticker":                 ticker,
            "event_type":             event_type,
            "articles":               cluster_articles,
            "representative_headline": rep_headline,
            "representative_source":  rep_source,
            "article_count":          len(cluster_articles),
            "sources":                all_sources,
        })

    return clusters


def clustering_agent(state: PipelineState) -> PipelineState:
    state["current_step"] = 4
    state["step_logs"].append("[Agent 4] Clustering articles by ticker then by event...")

    articles = state.get("clean_articles", [])
    if not articles:
        state["event_clusters"] = []
        state["step_logs"].append("[Agent 4] ✓ No articles to cluster")
        return state

    # Group by ticker
    by_ticker: dict[str, list[dict]] = {}
    for art in articles:
        by_ticker.setdefault(art.get("ticker", "UNKNOWN"), []).append(art)

    all_clusters: list[dict] = []
    offset = 0
    for ticker, ticker_articles in by_ticker.items():
        clusters = _cluster_ticker_articles(ticker, ticker_articles, state["openai_key"], offset)
        all_clusters.extend(clusters)
        offset += len(clusters)

    state["event_clusters"] = all_clusters
    state["step_logs"].append(
        f"[Agent 4] ✓ Formed {len(all_clusters)} event clusters "
        f"across {len(by_ticker)} tickers"
    )
    return state


In [ ]:
clustering_agent_4 = clustering_agent(filter_agent_3)
clustering_agent_4

In [ ]:
output_path = "agent_4_output.json"
with open(output_path, "w") as f:
    json.dump(clustering_agent_4, f, indent=4)

## Summarization Agent

In [ ]:
"""
Agent 5: ImpactSummarizationAgent
Converts each event cluster into a structured event card with two GPT-4o calls:
  1. Summarization  → tldr, key_facts, impact
  2. Verification   → confidence, uncertainty_flags

Injects market_context (price, analyst rating, volume) when available.

Output event card schema (per spec):
  cluster_id, ticker, event_type,
  tldr, key_facts, impact,
  confidence, uncertainty_flags,
  supporting_sources, source_urls, article_count
"""

import json
from utils.llm import call_openai, extract_json
from utils.state import PipelineState


SUMMARY_PROMPT = """You are a senior buy-side analyst writing investment intelligence for Singapore equity investors.

Event Type: {event_type}
Ticker: {ticker} — {company}

{market_context_section}

Source articles (up to 5):
{articles_text}

Write a structured event card. Return ONLY JSON:
{{
  "tldr":    "One crisp sentence summarising the event and its core financial significance",
  "key_facts": [
    "Specific quantitative fact 1 (with numbers where available)",
    "Specific quantitative fact 2",
    "Specific quantitative fact 3"
  ],
  "impact":  "2-3 sentences on investment significance — risks, opportunities, price catalysts, sector read-through",
  "sentiment": "bullish|bearish|neutral"
}}"""

VERIFY_PROMPT = """You are a fact-checker for a financial intelligence system.

Event Card to verify:
{card}

Source evidence (article headlines and snippets):
{evidence}

Check: Are the tldr and every key_fact directly supported by the source evidence above?
Return ONLY JSON:
{{
  "verified":             true,
  "confidence":           "high|medium|low",
  "confidence_adjustment":"none|downgrade",
  "uncertainty_flags":    ["list any claims not clearly supported by sources, or empty list"]
}}"""


def _build_market_context_section(ticker: str, market_context: dict) -> str:
    """Format market_context for a ticker into a prompt section."""
    ctx = market_context.get(ticker)
    if not ctx:
        return ""
    lines = ["Market Context (for reference):"]
    if ctx.get("last_price"):
        ccy = ctx.get("currency", "")
        p1d = ctx.get("price_change_1d", 0)
        p5d = ctx.get("price_change_5d", 0)
        lines.append(f"  Price: {ccy} {ctx['last_price']:.2f}  (1D: {p1d:+.1f}%,  5D: {p5d:+.1f}%)")
    if ctx.get("volume_ratio"):
        lines.append(f"  Volume ratio vs avg: {ctx['volume_ratio']:.1f}x")
    if ctx.get("analyst_rating"):
        tp = f"  Target: {ctx.get('currency','')} {ctx['target_price']:.2f}" if ctx.get("target_price") else ""
        lines.append(f"  Analyst consensus: {ctx['analyst_rating']}{tp}")
    if ctx.get("recent_eps_actual") is not None:
        lines.append(
            f"  Recent EPS: actual {ctx['recent_eps_actual']}  vs estimate {ctx.get('recent_eps_estimate','?')}"
        )
    if ctx.get("earnings_date"):
        lines.append(f"  Next earnings date: {ctx['earnings_date']}")
    return "\n".join(lines)


def _summarise_cluster(cluster: dict, openai_key: str, market_context: dict) -> dict:
    arts    = cluster.get("articles", [])[:5]   # spec: up to 5 per cluster
    ticker  = cluster.get("ticker", "")
    company = arts[0].get("company", ticker) if arts else ticker

    art_text = "\n---\n".join(
        f"Source: {a.get('source', '')}\nHeadline: {a.get('headline', '')}\n{a.get('snippet', '')}"
        for a in arts
    )
    ctx_section = _build_market_context_section(ticker, market_context)

    resp    = call_openai(
        SUMMARY_PROMPT.format(
            event_type=cluster.get("event_type", "general_news"),
            ticker=ticker,
            company=company,
            market_context_section=ctx_section,
            articles_text=art_text,
        ),
        openai_key,
    )
    return extract_json(resp)


def _verify_card(summary: dict, cluster: dict, openai_key: str) -> dict:
    arts     = cluster.get("articles", [])[:5]
    evidence = "\n".join(
        f"[{a.get('source','')}] {a.get('headline','')}: {a.get('snippet','')[:150]}"
        for a in arts
    )
    card_for_check = json.dumps({
        "tldr":      summary.get("tldr"),
        "key_facts": summary.get("key_facts"),
    })
    resp = call_openai(
        VERIFY_PROMPT.format(card=card_for_check, evidence=evidence),
        openai_key,
    )
    return extract_json(resp)


def summarization_agent(state: PipelineState) -> PipelineState:
    state["current_step"] = 5
    state["step_logs"].append("[Agent 5] Summarizing event clusters (GPT-4o)...")

    market_context = state.get("market_context", {})
    cards: list[dict] = []

    for cluster in state.get("event_clusters", []):
        arts = cluster.get("articles", [])
        ticker = cluster.get("ticker", "")

        # ── Fallback if no OpenAI key ─────────────────────────────────────────
        if not state.get("openai_key"):
            cards.append({
                "cluster_id":              cluster.get("cluster_id"),
                "ticker":                  ticker,
                "event_type":              cluster.get("event_type"),
                "representative_headline": cluster.get("representative_headline", ""),
                "representative_source":   cluster.get("representative_source", ""),
                "tldr":                    cluster.get("representative_headline", ""),
                "key_facts":               [],
                "impact":                  "",
                "sentiment":               "neutral",
                "confidence":              "low",
                "uncertainty_flags":       ["No OpenAI key — fallback to representative headline"],
                "supporting_sources":      cluster.get("sources", []),
                "source_urls":             [a.get("url", "") for a in arts],
                "article_count":           cluster.get("article_count", len(arts)),
            })
            continue

        # ── Call 1: Summarization ─────────────────────────────────────────────
        try:
            summary = _summarise_cluster(cluster, state["openai_key"], market_context)
        except Exception as e:
            summary = {
                "tldr":      cluster.get("representative_headline", ""),
                "key_facts": [],
                "impact":    "",
                "sentiment": "neutral",
                "uncertainty_flags": [f"Summarization failed: {e}"],
            }

        # ── Call 2: Verification ──────────────────────────────────────────────
        confidence        = "low"
        uncertainty_flags = list(summary.get("uncertainty_flags", []))
        try:
            verification = _verify_card(summary, cluster, state["openai_key"])
            confidence   = verification.get("confidence", "medium")
            # Downgrade if verifier says so
            if verification.get("confidence_adjustment") == "downgrade":
                if confidence == "high":
                    confidence = "medium"
                elif confidence == "medium":
                    confidence = "low"
            uncertainty_flags = list(set(
                uncertainty_flags + verification.get("uncertainty_flags", [])
            ))
        except Exception as e:
            uncertainty_flags.append(f"Verification failed: {e}")

        cards.append({
            "cluster_id":              cluster.get("cluster_id"),
            "ticker":                  ticker,
            "event_type":              cluster.get("event_type"),
            "representative_headline": cluster.get("representative_headline", ""),
            "representative_source":   cluster.get("representative_source", ""),
            "tldr":                    summary.get("tldr", ""),
            "key_facts":               summary.get("key_facts", []),
            "impact":                  summary.get("impact", ""),
            "sentiment":               summary.get("sentiment", "neutral"),
            "confidence":              confidence,
            "uncertainty_flags":       [f for f in uncertainty_flags if f],
            "supporting_sources":      cluster.get("sources", []),
            "source_urls":             [a.get("url", "") for a in arts],
            "article_count":           cluster.get("article_count", len(arts)),
        })

    state["event_cards"] = cards
    state["step_logs"].append(
        f"[Agent 5] ✓ Generated {len(cards)} event cards "
        f"({sum(1 for c in cards if c['confidence'] == 'high')} high-confidence)"
    )
    return state


In [ ]:
summarization_agent_5 = summarization_agent(clustering_agent_4)
summarization_agent_5

In [ ]:
output_path = "agent_5_output.json"
with open(output_path, "w") as f:
    json.dump(summarization_agent_5, f, indent=4)

## Ranking agent

In [ ]:
"""
Agent 6: ImportanceRankingAgent
Scores every event card 0-1 and assigns High / Medium / Low label.

Scoring formula (from spec):
  score = event_type_weight  × 0.40
        + corroboration_score × 0.25
        + novelty_score       × 0.20
        + credibility_score   × 0.15
        + confidence_adj

Signals:
  event_type_weight (40%) – fixed weights per event_type
  corroboration     (25%) – unique source count / 5, capped at 1.0
  novelty           (20%) – article_count / 8, capped at 1.0
  credibility       (15%) – avg credibility tier of supporting sources
  confidence_adj          – +0.05 high / 0 medium / -0.05 low
  volume_ratio bonus      – +0.05 if volume_ratio > 2.0 (from market_context)
  earnings_proximity bonus– +0.05 if within ±7 days of earnings_date

Thresholds:
  ≥ 0.70 → High
  ≥ 0.45 → Medium
  <  0.45 → Low

Adds to each event card:
  importance, importance_score (0-1), rank_overall, rank_per_ticker,
  scoring_signals { event_type_weight, corroboration_count, corroboration_score,
                    novelty_score, credibility_score, confidence_adj }
"""

from datetime import datetime, timezone, timedelta
from utils.state import PipelineState


# ── Event type weights (spec Table) ──────────────────────────────────────────
EVENT_TYPE_WEIGHTS: dict[str, float] = {
    "earnings_release":  0.95,
    "dividend":          0.75,
    "guidance_update":   0.85,
    "ma_announcement":   0.90,
    "regulatory_action": 0.80,
    "capital_action":    0.75,
    "leadership_change": 0.70,
    "litigation":        0.70,
    "product_launch":    0.60,
    "analyst_rating":    0.60,
    "general_news":      0.35,
    # Legacy / fallback keys
    "earnings":          0.95,
    "guidance":          0.85,
    "M&A":               0.90,
    "regulation":        0.80,
    "macro":             0.50,
    "market_trend":      0.40,
    "other":             0.35,
}

# Source credibility lookup (must match retrieval_agent)
SOURCE_CREDIBILITY: dict[str, float] = {
    "sgx":           1.00, "mas":           1.00,
    "business times":0.85, "straits times": 0.85,
    "reuters":       0.85, "bloomberg":     0.85,
    "cna":           0.70, "nikkei":        0.70, "cnbc": 0.70,
    "yahoo":         0.55, "singapore business review": 0.55,
}
DEFAULT_CRED = 0.55


def _source_credibility(source_name: str) -> float:
    sl = source_name.lower()
    for key, score in SOURCE_CREDIBILITY.items():
        if key in sl:
            return score
    return DEFAULT_CRED


def _confidence_adj(confidence: str) -> float:
    return {"high": 0.05, "medium": 0.0, "low": -0.05}.get(confidence, 0.0)


def _volume_bonus(ticker: str, market_context: dict) -> float:
    ctx = market_context.get(ticker, {})
    return 0.05 if (ctx.get("volume_ratio") or 0) > 2.0 else 0.0


def _earnings_proximity_bonus(ticker: str, market_context: dict) -> float:
    ctx = market_context.get(ticker, {})
    ed  = ctx.get("earnings_date")
    if not ed:
        return 0.0
    try:
        earnings_dt = datetime.fromisoformat(ed)
        if earnings_dt.tzinfo is None:
            earnings_dt = earnings_dt.replace(tzinfo=timezone.utc)
        delta = abs((earnings_dt - datetime.now(timezone.utc)).days)
        return 0.05 if delta <= 7 else 0.0
    except ValueError:
        return 0.0


def _score_card(card: dict, market_context: dict) -> tuple[float, dict]:
    """Return (raw_score, scoring_signals_dict)."""
    ticker = card.get("ticker", "")

    # Event type weight
    et     = card.get("event_type", "general_news")
    et_w   = EVENT_TYPE_WEIGHTS.get(et, 0.35)

    # Corroboration: unique source count / 5
    sources       = card.get("supporting_sources", [])
    corr_count    = len(set(sources))
    corr_score    = min(corr_count / 5, 1.0)

    # Novelty: article_count / 8
    novelty_score = min(card.get("article_count", 1) / 8, 1.0)

    # Credibility: average tier score across supporting sources
    cred_scores   = [_source_credibility(s) for s in sources] if sources else [DEFAULT_CRED]
    cred_score    = sum(cred_scores) / len(cred_scores)

    # Confidence adjustment
    conf_adj      = _confidence_adj(card.get("confidence", "medium"))

    # Bonuses from market context
    vol_bonus     = _volume_bonus(ticker, market_context)
    earn_bonus    = _earnings_proximity_bonus(ticker, market_context)

    raw_score = (
        et_w        * 0.40 +
        corr_score  * 0.25 +
        novelty_score * 0.20 +
        cred_score  * 0.15 +
        conf_adj    +
        vol_bonus   +
        earn_bonus
    )
    raw_score = min(max(raw_score, 0.0), 1.0)

    signals = {
        "event_type_weight":   et_w,
        "corroboration_count": corr_count,
        "corroboration_score": round(corr_score, 3),
        "novelty_score":       round(novelty_score, 3),
        "credibility_score":   round(cred_score, 3),
        "confidence_adj":      conf_adj,
        "volume_bonus":        vol_bonus,
        "earnings_proximity_bonus": earn_bonus,
    }
    return round(raw_score, 4), signals


def _label(score: float) -> str:
    if score >= 0.70:
        return "High"
    if score >= 0.45:
        return "Medium"
    return "Low"


def ranking_agent(state: PipelineState) -> PipelineState:
    state["current_step"] = 6
    state["step_logs"].append("[Agent 6] Scoring and ranking events...")

    cards          = state.get("event_cards", [])
    market_context = state.get("market_context", {})

    # Score every card
    for card in cards:
        score, signals = _score_card(card, market_context)
        card["importance_score"]   = score
        card["importance"]         = _label(score)
        card["scoring_signals"]    = signals

    # Overall rank (descending score)
    cards.sort(key=lambda c: c["importance_score"], reverse=True)
    for i, card in enumerate(cards):
        card["rank_overall"] = i + 1

    # Per-ticker rank
    ticker_counters: dict[str, int] = {}
    for card in cards:
        t = card.get("ticker", "")
        ticker_counters[t] = ticker_counters.get(t, 0) + 1
        card["rank_per_ticker"] = ticker_counters[t]

    state["ranked_events"] = cards
    high   = sum(1 for c in cards if c["importance"] == "High")
    medium = sum(1 for c in cards if c["importance"] == "Medium")
    low    = sum(1 for c in cards if c["importance"] == "Low")
    state["step_logs"].append(
        f"[Agent 6] ✓ Ranked {len(cards)} events — "
        f"{high} High / {medium} Medium / {low} Low"
    )
    return state


In [ ]:
ranking_agent_6 = ranking_agent(summarization_agent_5)
ranking_agent_6

In [ ]:
output_path = "agent_6_output.json"
with open(output_path, "w") as f:
    json.dump(ranking_agent_6, f, indent=4)

## Manus State

In [ ]:
"""
LangGraph State Schema for Financial News Intelligence Pipeline
Defines the shared state that flows through all 7 agents
Follows the detailed data flow specification with proper article, cluster, and event card schemas
"""

from typing import TypedDict, Optional, List, Dict, Any
from dataclasses import dataclass, field
from datetime import datetime


@dataclass
class Article:
    """
    Normalized article schema from data collection.
    Every article produced by news retrieval has this structure.
    """
    ticker: str                          # SGX ticker (e.g., D05.SI) or MACRO
    company: str                         # Full company name
    headline: str                        # Article title
    snippet: str                         # Short description or article excerpt
    url: str                            # Canonical article URL
    source: str                         # Data source name (e.g., SGX Announcements, Reuters)
    published_at: str                   # ISO 8601 UTC timestamp
    query_type: str                     # company / industry / macro / sentiment
    raw: Dict[str, Any] = field(default_factory=dict)  # Original API/feed response


@dataclass
class EventCluster:
    """
    Event cluster schema - groups related articles describing the same real-world event.
    Output from Agent 4 (EventClusteringAgent).
    """
    cluster_id: str                     # Unique cluster ID (e.g., D05.SI_c000)
    ticker: str                         # SGX ticker
    event_type: str                     # earnings_release, dividend, guidance_update, etc.
    articles: List[Article] = field(default_factory=list)  # Raw articles in cluster
    representative_headline: str = ""   # Headline of highest-tier source article
    representative_source: str = ""     # Source of representative article
    article_count: int = 0              # Number of articles in cluster
    sources: List[str] = field(default_factory=list)  # Unique sources in cluster


@dataclass
class EventCard:
    """
    Event card schema - structured, human-readable summary of an event.
    Output from Agent 5 (ImpactSummarizationAgent) and Agent 6 (ImportanceRankingAgent).
    """
    # Core event information
    cluster_id: str                     # Reference to source cluster
    ticker: str                         # SGX ticker
    event_type: str                     # Event classification
    
    # Summarization (from Agent 5)
    tldr: str                          # One-line summary
    key_facts: List[str] = field(default_factory=list)  # Bullet points of key facts
    impact: str = ""                   # Impact explanation
    confidence: str = "medium"         # high / medium / low
    uncertainty_flags: List[str] = field(default_factory=list)  # Flags for uncertain facts
    
    # Source information
    supporting_sources: List[str] = field(default_factory=list)  # Source names
    source_urls: List[str] = field(default_factory=list)  # Article URLs
    article_count: int = 0             # Number of articles in cluster
    
    # Importance ranking (from Agent 6)
    importance: str = "Medium"         # High / Medium / Low
    importance_score: float = 0.0      # 0-1 score
    rank_overall: int = 0              # Overall rank across all events
    rank_per_ticker: int = 0           # Rank within this ticker
    
    # Scoring signals (from Agent 6)
    scoring_signals: Dict[str, Any] = field(default_factory=dict)  # Detailed scoring breakdown


@dataclass
class MarketContext:
    """
    Quantitative market context per ticker.
    Output from Agent 1b (MarketDataAgent).
    Used by downstream agents for context and scoring.
    """
    ticker: str
    last_price: float
    currency: str
    price_change_1d: float             # % change in last 1 day
    price_change_5d: float             # % change in last 5 days
    volume_ratio: float                # Current volume / average volume
    analyst_rating: str                # Buy / Hold / Sell
    target_price: float
    earnings_date: Optional[str]       # ISO 8601 date
    recent_eps_actual: float
    recent_eps_estimate: float
    fetched_at: str                    # ISO 8601 UTC timestamp


class PipelineState(TypedDict):
    """
    Shared state object that flows through the LangGraph pipeline.
    Each agent reads from and writes to this state.
    """
    # Input parameters
    tickers: List[str]                 # List of SGX tickers to analyze
    user_id: int                       # User ID for storage
    run_id: int                        # Pipeline run ID for tracking
    llm_provider: str                  # openai or gemini
    
    # Agent 1: WatchlistContextAgent
    ticker_metadata: Dict[str, Dict[str, Any]]  # {ticker: {name, sector, ...}}
    
    # Agent 1b: MarketDataAgent
    market_context: Dict[str, MarketContext]  # {ticker: MarketContext}
    
    # Agent 2: NewsRetrievalAgent
    raw_articles: List[Article]        # All collected articles
    articles_by_ticker: Dict[str, List[Article]]  # {ticker: [articles]}
    
    # Agent 3: NoiseFilterAgent
    filtered_articles: List[Article]   # After hard filters and semantic dedup
    deduplication_metadata: Dict[str, Any]  # Dedup statistics
    
    # Agent 4: EventClusteringAgent
    event_clusters: List[EventCluster]  # Grouped articles by event
    clustering_metadata: Dict[str, Any]  # Clustering statistics
    
    # Agent 5: ImpactSummarizationAgent
    event_cards: List[EventCard]       # Events with summaries
    summarization_metadata: Dict[str, Any]  # Summarization statistics
    
    # Agent 6: ImportanceRankingAgent
    ranked_events: List[EventCard]     # Sorted by importance score
    ranking_metadata: Dict[str, Any]   # Ranking statistics
    
    # Agent 7: NotificationAgent
    digest_json: str                   # JSON digest output
    digest_html: str                   # HTML email output
    digest_subject: str                # Email subject
    
    # Metadata & Error Handling
    current_agent: str                 # Current agent name
    progress: int                      # 0-100
    status: str                        # running / completed / failed
    error_message: str                 # Error details if failed
    agent_timings: Dict[str, float]    # Execution time per agent
    started_at: Optional[datetime]     # Pipeline start time
    completed_at: Optional[datetime]   # Pipeline completion time


def create_initial_state(
    tickers: List[str],
    user_id: int,
    run_id: int,
    llm_provider: str = "openai",
) -> PipelineState:
    """
    Initialize the pipeline state with default values.
    
    Args:
        tickers: List of SGX tickers to analyze
        user_id: User ID for storage
        run_id: Pipeline run ID
        llm_provider: LLM provider (openai or gemini)
    
    Returns:
        Initialized PipelineState
    """
    return {
        "tickers": tickers,
        "user_id": user_id,
        "run_id": run_id,
        "llm_provider": llm_provider,
        "ticker_metadata": {},
        "market_context": {},
        "raw_articles": [],
        "articles_by_ticker": {},
        "filtered_articles": [],
        "deduplication_metadata": {},
        "event_clusters": [],
        "clustering_metadata": {},
        "event_cards": [],
        "summarization_metadata": {},
        "ranked_events": [],
        "ranking_metadata": {},
        "digest_json": "",
        "digest_html": "",
        "digest_subject": "",
        "current_agent": "initializing",
        "progress": 0,
        "status": "running",
        "error_message": "",
        "agent_timings": {},
        "started_at": None,
        "completed_at": None,
    }


In [ ]:
import time
import yfinance as yf
import feedparser
def agent_1_watchlist_context(state: PipelineState) -> PipelineState:
    """
    Agent 1: Watchlist & Context Agent
    Resolves tickers to company metadata and aliases.
    """
    start_time = time.time()
    state["current_agent"] = "Agent 1: Watchlist & Context"
    state["progress"] = 5
    
    try:
        ticker_metadata = {}
        
        for ticker in state["tickers"]:
            try:
                stock = yf.Ticker(ticker)
                info = stock.info
                
                ticker_metadata[ticker] = {
                    "name": info.get("longName", ticker),
                    "sector": info.get("sector", "Unknown"),
                    "industry": info.get("industry", "Unknown"),
                    "market_cap": info.get("marketCap", 0),
                    "pe_ratio": info.get("trailingPE", None),
                    "dividend_yield": info.get("dividendYield", None),
                }
            except Exception as e:
                ticker_metadata[ticker] = {
                    "name": ticker,
                    "sector": "Unknown",
                    "industry": "Unknown",
                }
        
        state["ticker_metadata"] = ticker_metadata
        
    except Exception as e:
        state["error_message"] = f"Agent 1 error: {str(e)}"
        state["status"] = "failed"
    
    state["agent_timings"]["agent_1"] = time.time() - start_time
    return state


# ============================================================================
# AGENT 1b: MarketDataAgent
# ============================================================================

def agent_1b_market_data(state: PipelineState) -> PipelineState:
    """
    Agent 1b: Market Data Agent
    Fetches quantitative market context (price, volume, analyst ratings, etc.)
    """
    start_time = time.time()
    state["current_agent"] = "Agent 1b: Market Data"
    state["progress"] = 10
    
    try:
        market_context = {}
        
        for ticker in state["tickers"]:
            try:
                stock = yf.Ticker(ticker)
                info = stock.info
                hist = stock.history(period="5d")
                
                # Calculate price changes
                if len(hist) >= 2:
                    price_change_1d = ((hist.iloc[-1]["Close"] - hist.iloc[-2]["Close"]) / hist.iloc[-2]["Close"]) * 100
                    price_change_5d = ((hist.iloc[-1]["Close"] - hist.iloc[0]["Close"]) / hist.iloc[0]["Close"]) * 100
                else:
                    price_change_1d = 0.0
                    price_change_5d = 0.0
                
                # Calculate volume ratio
                avg_volume = info.get("averageVolume", 1)
                current_volume = info.get("volume", avg_volume)
                volume_ratio = current_volume / avg_volume if avg_volume > 0 else 1.0
                
                market_context[ticker] = MarketContext(
                    ticker=ticker,
                    last_price=info.get("currentPrice", 0.0),
                    currency=info.get("currency", "USD"),
                    price_change_1d=price_change_1d,
                    price_change_5d=price_change_5d,
                    volume_ratio=volume_ratio,
                    analyst_rating=info.get("recommendationKey", "hold").title(),
                    target_price=info.get("targetMeanPrice", 0.0),
                    earnings_date=None,  # Would need additional API call
                    recent_eps_actual=info.get("trailingEps", 0.0),
                    recent_eps_estimate=info.get("epsTrailingTwelveMonths", 0.0),
                    fetched_at=datetime.now().isoformat() + "Z",
                )
            except Exception as e:
                # Fallback with zeros
                market_context[ticker] = MarketContext(
                    ticker=ticker,
                    last_price=0.0,
                    currency="USD",
                    price_change_1d=0.0,
                    price_change_5d=0.0,
                    volume_ratio=1.0,
                    analyst_rating="Hold",
                    target_price=0.0,
                    earnings_date=None,
                    recent_eps_actual=0.0,
                    recent_eps_estimate=0.0,
                    fetched_at=datetime.now().isoformat() + "Z",
                )
        
        state["market_context"] = market_context
        
    except Exception as e:
        state["error_message"] = f"Agent 1b error: {str(e)}"
        state["status"] = "failed"
    
    state["agent_timings"]["agent_1b"] = time.time() - start_time
    return state

"""
Enhanced Agent 2: News Retrieval Agent with Multi-Source Support
Fetches articles from 9+ financial news sources in parallel
"""

import time
import feedparser
from datetime import datetime
from typing import List, Dict, Any
import yfinance as yf
from concurrent.futures import ThreadPoolExecutor, as_completed




def fetch_from_yahoo_finance(ticker: str, company_name: str) -> List[Article]:
    """Fetch from Yahoo Finance RSS"""
    articles = []
    try:
        feed_url = f"https://feeds.finance.yahoo.com/rss/2.0/headline?s={ticker}"
        feed = feedparser.parse(feed_url)
        
        for entry in feed.entries[:5]:
            article = Article(
                ticker=ticker,
                company=company_name,
                headline=entry.get("title", ""),
                snippet=entry.get("summary", "")[:200],
                url=entry.get("link", ""),
                source="Yahoo Finance",
                published_at=entry.get("published", datetime.now().isoformat()),
                query_type="company",
                raw={},
            )
            articles.append(article)
    except Exception as e:
        print(f"Yahoo Finance error for {ticker}: {e}")
    
    return articles


def fetch_from_reuters(ticker: str, company_name: str) -> List[Article]:
    """Fetch from Reuters RSS"""
    articles = []
    try:
        feed_url = "https://feeds.reuters.com/reuters/businessNews"
        feed = feedparser.parse(feed_url)
        
        for entry in feed.entries[:10]:
            title = entry.get("title", "").lower()
            summary = entry.get("summary", "").lower()
            
            if ticker.lower() in title or ticker.lower() in summary:
                article = Article(
                    ticker=ticker,
                    company=company_name,
                    headline=entry.get("title", ""),
                    snippet=entry.get("summary", "")[:200],
                    url=entry.get("link", ""),
                    source="Reuters",
                    published_at=entry.get("published", datetime.now().isoformat()),
                    query_type="company",
                    raw={},
                )
                articles.append(article)
    except Exception as e:
        print(f"Reuters error for {ticker}: {e}")
    
    return articles


def fetch_from_bloomberg(ticker: str, company_name: str) -> List[Article]:
    """Fetch from Bloomberg RSS"""
    articles = []
    try:
        feed_url = "https://feeds.bloomberg.com/markets/news.rss"
        feed = feedparser.parse(feed_url)
        
        for entry in feed.entries[:10]:
            title = entry.get("title", "").lower()
            summary = entry.get("summary", "").lower()
            
            if ticker.lower() in title or ticker.lower() in summary:
                article = Article(
                    ticker=ticker,
                    company=company_name,
                    headline=entry.get("title", ""),
                    snippet=entry.get("summary", "")[:200],
                    url=entry.get("link", ""),
                    source="Bloomberg",
                    published_at=entry.get("published", datetime.now().isoformat()),
                    query_type="company",
                    raw={},
                )
                articles.append(article)
    except Exception as e:
        print(f"Bloomberg error for {ticker}: {e}")
    
    return articles


def fetch_from_cnbc(ticker: str, company_name: str) -> List[Article]:
    """Fetch from CNBC RSS"""
    articles = []
    try:
        feed_url = "https://feeds.cnbc.com/id/100003114/"
        feed = feedparser.parse(feed_url)
        
        for entry in feed.entries[:10]:
            title = entry.get("title", "").lower()
            summary = entry.get("summary", "").lower()
            
            if ticker.lower() in title or ticker.lower() in summary:
                article = Article(
                    ticker=ticker,
                    company=company_name,
                    headline=entry.get("title", ""),
                    snippet=entry.get("summary", "")[:200],
                    url=entry.get("link", ""),
                    source="CNBC",
                    published_at=entry.get("published", datetime.now().isoformat()),
                    query_type="company",
                    raw={},
                )
                articles.append(article)
    except Exception as e:
        print(f"CNBC error for {ticker}: {e}")
    
    return articles


def fetch_from_marketwatch(ticker: str, company_name: str) -> List[Article]:
    """Fetch from MarketWatch RSS"""
    articles = []
    try:
        feed_url = "https://feeds.marketwatch.com/marketwatch/topstories/"
        feed = feedparser.parse(feed_url)
        
        for entry in feed.entries[:10]:
            title = entry.get("title", "").lower()
            summary = entry.get("summary", "").lower()
            
            if ticker.lower() in title or ticker.lower() in summary:
                article = Article(
                    ticker=ticker,
                    company=company_name,
                    headline=entry.get("title", ""),
                    snippet=entry.get("summary", "")[:200],
                    url=entry.get("link", ""),
                    source="MarketWatch",
                    published_at=entry.get("published", datetime.now().isoformat()),
                    query_type="company",
                    raw={},
                )
                articles.append(article)
    except Exception as e:
        print(f"MarketWatch error for {ticker}: {e}")
    
    return articles


def fetch_from_seeking_alpha(ticker: str, company_name: str) -> List[Article]:
    """Fetch from Seeking Alpha RSS"""
    articles = []
    try:
        feed_url = "https://seekingalpha.com/feed.xml"
        feed = feedparser.parse(feed_url)
        
        for entry in feed.entries[:10]:
            title = entry.get("title", "").lower()
            summary = entry.get("summary", "").lower()
            
            if ticker.lower() in title or ticker.lower() in summary:
                article = Article(
                    ticker=ticker,
                    company=company_name,
                    headline=entry.get("title", ""),
                    snippet=entry.get("summary", "")[:200],
                    url=entry.get("link", ""),
                    source="Seeking Alpha",
                    published_at=entry.get("published", datetime.now().isoformat()),
                    query_type="company",
                    raw={},
                )
                articles.append(article)
    except Exception as e:
        print(f"Seeking Alpha error for {ticker}: {e}")
    
    return articles


def fetch_from_financial_times(ticker: str, company_name: str) -> List[Article]:
    """Fetch from Financial Times RSS"""
    articles = []
    try:
        feed_url = "https://feeds.ft.com/markets"
        feed = feedparser.parse(feed_url)
        
        for entry in feed.entries[:10]:
            title = entry.get("title", "").lower()
            summary = entry.get("summary", "").lower()
            
            if ticker.lower() in title or ticker.lower() in summary:
                article = Article(
                    ticker=ticker,
                    company=company_name,
                    headline=entry.get("title", ""),
                    snippet=entry.get("summary", "")[:200],
                    url=entry.get("link", ""),
                    source="Financial Times",
                    published_at=entry.get("published", datetime.now().isoformat()),
                    query_type="company",
                    raw={},
                )
                articles.append(article)
    except Exception as e:
        print(f"Financial Times error for {ticker}: {e}")
    
    return articles


def fetch_from_tradingview(ticker: str, company_name: str) -> List[Article]:
    """Fetch from TradingView RSS"""
    articles = []
    try:
        # TradingView ticker format may differ
        tv_ticker = ticker.replace(".", "%2E")
        feed_url = f"https://feeds.tradingview.com/stocks/symbol/{tv_ticker}/news"
        feed = feedparser.parse(feed_url)
        
        for entry in feed.entries[:5]:
            article = Article(
                ticker=ticker,
                company=company_name,
                headline=entry.get("title", ""),
                snippet=entry.get("summary", "")[:200],
                url=entry.get("link", ""),
                source="TradingView",
                published_at=entry.get("published", datetime.now().isoformat()),
                query_type="company",
                raw={},
            )
            articles.append(article)
    except Exception as e:
        print(f"TradingView error for {ticker}: {e}")
    
    return articles


def fetch_from_yfinance_news(ticker: str, company_name: str) -> List[Article]:
    """Fetch from yfinance built-in news"""
    articles = []
    try:
        stock = yf.Ticker(ticker)
        news = stock.news
        
        if news:
            for news_item in news[:5]:
                article = Article(
                    ticker=ticker,
                    company=company_name,
                    headline=news_item.get("title", ""),
                    snippet=news_item.get("summary", "")[:200] if "summary" in news_item else "",
                    url=news_item.get("link", ""),
                    source="Company News",
                    published_at=datetime.fromtimestamp(
                        news_item.get("providerPublishTime", 0)
                    ).isoformat() if news_item.get("providerPublishTime") else datetime.now().isoformat(),
                    query_type="company",
                    raw={},
                )
                articles.append(article)
    except Exception as e:
        print(f"yfinance news error for {ticker}: {e}")
    
    return articles


def fetch_from_investing_com(ticker: str, company_name: str) -> List[Article]:
    """Fetch from Investing.com RSS"""
    articles = []
    try:
        feed_url = "https://feeds.investing.com/equities/news"
        feed = feedparser.parse(feed_url)
        
        for entry in feed.entries[:10]:
            title = entry.get("title", "").lower()
            summary = entry.get("summary", "").lower()
            
            if ticker.lower() in title or ticker.lower() in summary:
                article = Article(
                    ticker=ticker,
                    company=company_name,
                    headline=entry.get("title", ""),
                    snippet=entry.get("summary", "")[:200],
                    url=entry.get("link", ""),
                    source="Investing.com",
                    published_at=entry.get("published", datetime.now().isoformat()),
                    query_type="company",
                    raw={},
                )
                articles.append(article)
    except Exception as e:
        print(f"Investing.com error for {ticker}: {e}")
    
    return articles


def agent_2_news_retrieval_multisource(state: PipelineState) -> PipelineState:
    """
    Enhanced Agent 2: News Retrieval Agent
    Fetches articles from 9+ financial news sources in parallel using ThreadPoolExecutor
    Returns normalized Article objects.
    """
    start_time = time.time()
    state["current_agent"] = "Agent 2: News Retrieval (Multi-Source)"
    state["progress"] = 25
    
    try:
        raw_articles = []
        articles_by_ticker = {}
        
        # Define all source fetchers
        source_fetchers = [
            fetch_from_yahoo_finance,
            fetch_from_reuters,
            fetch_from_bloomberg,
            fetch_from_cnbc,
            fetch_from_marketwatch,
            fetch_from_seeking_alpha,
            fetch_from_financial_times,
            fetch_from_tradingview,
            fetch_from_yfinance_news,
            fetch_from_investing_com,
        ]
        
        # Fetch from all sources in parallel
        for ticker in state["tickers"]:
            articles_by_ticker[ticker] = []
            company_name = state["ticker_metadata"].get(ticker, {}).get("name", ticker)
            
            # Use ThreadPoolExecutor for parallel fetching
            with ThreadPoolExecutor(max_workers=1) as executor:
                futures = {
                    executor.submit(fetcher, ticker, company_name): fetcher.__name__
                    for fetcher in source_fetchers
                }
                
                for future in as_completed(futures):
                    try:
                        articles = future.result()
                        raw_articles.extend(articles)
                        articles_by_ticker[ticker].extend(articles)
                    except Exception as e:
                        print(f"Error in {futures[future]}: {e}")
        
        state["raw_articles"] = raw_articles
        state["articles_by_ticker"] = articles_by_ticker
        
        # Log retrieval stats
        unique_sources = len(set(a.source for a in raw_articles))
        state["retrieval_metadata"] = {
            "total_articles": len(raw_articles),
            "unique_sources": unique_sources,
            "articles_per_ticker": {
                ticker: len(articles) for ticker, articles in articles_by_ticker.items()
            },
        }
        
    except Exception as e:
        state["error_message"] = f"Agent 2 error: {str(e)}"
        state["status"] = "failed"
    
    state["agent_timings"]["agent_2"] = time.time() - start_time
    return state


# # ============================================================================
# # USAGE EXAMPLE
# # ============================================================================

if __name__ == "__main__":
    #from langgraph_pipeline.state import create_initial_state
    #
    # Create initial state
    state = create_initial_state(
        tickers=["AAPL", "NVDA", "TSLA"],
        user_id=1,
        run_id=1,
        llm_provider="openai"
    )
    
    # Populate ticker metadata first
    state["ticker_metadata"] = {
        "AAPL": {"name": "Apple Inc."},
        "NVDA": {"name": "NVIDIA Corporation"},
        "TSLA": {"name": "Tesla Inc."},
    }
    
    # Run Agent 2
    state = agent_2_news_retrieval_multisource(state)
    
    print(f"✓ Multi-source retrieval completed")
    print(f"Total articles: {len(state['raw_articles'])}")
    print(f"Unique sources: {state['retrieval_metadata']['unique_sources']}")
    print(f"Articles per ticker: {state['retrieval_metadata']['articles_per_ticker']}")
    print(f"\nSources found:")
    for source in sorted(set(a.source for a in state['raw_articles'])):
        count = sum(1 for a in state['raw_articles'] if a.source == source)
        print(f"  - {source}: {count} articles")

    
    #state["agent_timings"]["agent_2"] = time.time() - start_time
    #return state
    state

In [ ]:
state = create_initial_state(['AA'], 0, 0) 


manus_agent1_watchlist = agent_1_watchlist_context(state)
manus_agent1_watchlist

In [ ]:
manus_agent1b_watchlist = agent_1b_market_data(manus_agent1_watchlist)
manus_agent1b_watchlist

In [ ]:
manus_agent2_watchlist = agent_2_news_retrieval_multisource(manus_agent1b_watchlist)
manus_agent2_watchlist